# Image Captioning (BLIP2 + Local MT)

เวอร์ชันนี้ใช้ `BLIP2` สร้างคำบรรยายภาษาอังกฤษ แล้วแปลเป็นไทยด้วยโมเดลแปลภาษา local (ไม่ใช้ third-party API)

## 0) Setup (Colab + Packages)

In [ ]:
# Run once in a fresh Colab environment
!pip -q install torch torchvision transformers sentencepiece sacremoses sacrebleu pillow pandas tqdm accelerate kagglehub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.2 MB/s eta 0:00:00


## 0.5) Login Kaggle And Download Competition Data

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

super_ai_engineer_ss_6_thai_language_image_captioning_path = kagglehub.competition_download(
    "super-ai-engineer-ss-6-thai-language-image-captioning"
)

print(super_ai_engineer_ss_6_thai_language_image_captioning_path)


100%|██████████| 1.75G/1.75G [00:19<00:00, 95.6MB/s]

Extracting files...


/root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-thai-language-image-captioning


## 1) Config (Balanced Colab Preset)

In [ ]:
import os

DEFAULT_DATA_ROOT = "/root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-thai-language-image-captioning"
DATA_ROOT = globals().get(
    "super_ai_engineer_ss_6_thai_language_image_captioning_path",
    DEFAULT_DATA_ROOT,
)

CFG = {
    # Data
    "TEST_DIR": "/root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/test/test",
    "SAMPLE_SUBMISSION_PATH": "/root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/sample_submission.csv",
    "OUTPUT_PATH": "submission_balanced_colab.csv",

    # Balanced preset: slower than fast mode, much better than 0.28 target
    "CAPTION_MODEL": "Salesforce/blip-image-captioning-base",
    "CAPTION_BATCH_SIZE": 8,
    "CAPTION_NUM_RETURN_SEQUENCES": 1,
    "USE_SAMPLING_CANDIDATES": False,
    "USE_PROMPT_CANDIDATE": True,
    "CAPTION_PROMPT": "a photo of",
    "CAPTION_MAX_NEW_TOKENS": 30,
    "CAPTION_MIN_NEW_TOKENS": 5,
    "CAPTION_NUM_BEAMS": 4,
    "CAPTION_LENGTH_PENALTY": 1.0,
    "CAPTION_REPETITION_PENALTY": 1.1,
    "CAPTION_NO_REPEAT_NGRAM_SIZE": 3,
    "CAPTION_TOP_P": 0.92,
    "CAPTION_TEMPERATURE": 0.9,

    # Translation preset
    "MT_MODEL": "facebook/nllb-200-distilled-1.3B",
    "SRC_LANG": "eng_Latn",
    "TGT_LANG": "tha_Thai",
    "TRANSLATE_BATCH_SIZE": 48,
    "MT_MAX_NEW_TOKENS": 56,
    "MT_NUM_BEAMS": 2,
    "MT_LENGTH_PENALTY": 1.0,
    "MT_NO_REPEAT_NGRAM_SIZE": 3,

    # Caption cleanup
    "MAX_THAI_CHARS": 70,
    "MIN_THAI_CHARS": 6,
    "KEEP_END_PUNCT": False,
    "OVERWRITE_SUBMISSION": True,
    "FALLBACK_THAI_CAPTION": "ภาพถ่ายที่มีวัตถุและฉาก",

    # Runtime
    "SEED": 42,
    "BATCH_PRINT_EVERY": 20,
    "USE_FP16_ON_CUDA": True,
}

CFG["TEST_DIR"] = os.path.join(DATA_ROOT, "test", "test")
CFG["SAMPLE_SUBMISSION_PATH"] = os.path.join(DATA_ROOT, "sample_submission.csv")

assert os.path.exists(CFG["TEST_DIR"]), f"Missing test dir: {CFG['TEST_DIR']}"
assert os.path.exists(CFG["SAMPLE_SUBMISSION_PATH"]), f"Missing sample submission: {CFG['SAMPLE_SUBMISSION_PATH']}"
print("DATA_ROOT:", DATA_ROOT)
print(CFG)


DATA_ROOT: /root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-thai-language-image-captioning
{'TEST_DIR': '/root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/test/test', 'SAMPLE_SUBMISSION_PATH': '/root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/sample_submission.csv', 'OUTPUT_PATH': 'submission_balanced_colab.csv', 'CAPTION_MODEL': 'Salesforce/blip-image-captioning-base', 'CAPTION_BATCH_SIZE': 8, 'CAPTION_NUM_RETURN_SEQUENCES': 1, 'USE_SAMPLING_CANDIDATES': False, 'USE_PROMPT_CANDIDATE': True, 'CAPTION_PROMPT': 'a photo of', 'CAPTION_MAX_NEW_TOKENS': 30, 'CAPTION_MIN_NEW_TOKENS': 5, 'CAPTION_NUM_BEAMS': 4, 'CAPTION_LENGTH_PENALTY': 1.0, 'CAPTION_REPETITION_PENALTY': 1.1, 'CAPTION_NO_REPEAT_NGRAM_SIZE': 3, 'CAPTION_TOP_P': 0.92, 'CAPTION_TEMPERATURE': 0.9, 'MT_MODEL': 'facebook/nllb-200-distilled-1.3B', 'SRC_LANG': 'eng_Latn', 'TGT_LANG': 'tha_Thai', 'TRANSLATE_BATCH_SIZE': 48, 'MT_MAX_NEW_TOKENS': 

## 2) Imports + Utilities

In [ ]:
import gc
import re
import random
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BlipForConditionalGeneration,
    BlipProcessor,
)


TRAILING_STOPWORDS = {
    "a", "an", "the", "of", "with", "and", "in", "on", "at", "to", "from", "by", "for", "near"
}
HUMAN_WORDS = {
    "man", "woman", "person", "people", "child", "children", "boy", "girl", "monk", "farmer"
}
FOOD_WORDS = {
    "food", "soup", "rice", "noodle", "noodles", "salad", "curry", "vegetable", "vegetables", "meat",
    "shrimp", "chicken", "fish", "pasta", "dessert", "drink", "coffee", "tea", "beer", "sauce"
}
ENGLISH_LEFTOVER_RE = re.compile(r"\b[a-zA-Z][a-zA-Z'/-]*\b")


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def remove_immediate_repetition(text: str) -> str:
    tokens = text.split()
    if not tokens:
        return text
    compact = [tokens[0]]
    for token in tokens[1:]:
        if token != compact[-1]:
            compact.append(token)
    return " ".join(compact)


def chunked(items: List[Path], size: int):
    for start in range(0, len(items), size):
        yield items[start:start + size]


def repair_english_caption(text: str) -> str:
    text = (text or "").strip().lower()
    text = text.replace(" ' s", "'s")
    text = text.replace(" ,", ",")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(
        r"\b(" + "|".join(re.escape(word) for word in TRAILING_STOPWORDS) + r")(?: \1)+\b",
        r"\1",
        text,
    )
    text = remove_immediate_repetition(text)
    tokens = text.split()
    while tokens and tokens[-1] in TRAILING_STOPWORDS:
        tokens.pop()
    while tokens and len(tokens[-1]) == 1 and tokens[-1].isalpha():
        tokens.pop()
    text = " ".join(tokens).strip(" ,.;:-")
    return text


def caption_quality_score(text: str) -> float:
    tokens = text.split()
    if not tokens:
        return -1e9

    score = 0.0
    token_count = len(tokens)
    score += min(token_count, 14) * 0.25

    if token_count < 4:
        score -= 2.5
    if tokens[-1] in TRAILING_STOPWORDS:
        score -= 4.0
    if text.startswith("there is") or text.startswith("there are"):
        score -= 1.5
    if re.search(r"\b(\w+)(?: \1){1,}\b", text):
        score -= 2.0
    if any(ch.isdigit() for ch in text):
        score -= 0.4
    if " ," in text or " ." in text:
        score -= 0.5
    if any(noun in text for noun in ["bird", "flower", "tree", "river", "plate", "bowl", "temple", "statue", "frog"]):
        score += 0.5
    if token_count > 18:
        score -= (token_count - 18) * 0.1

    return score


def is_food_caption(text: str) -> bool:
    return any(word in text for word in FOOD_WORDS)


def collapse_duplicate_tail(text: str, target: str) -> str:
    text = text.replace(f"อยู่บน{target}", "")
    text = text.replace(f"บน{target}", "")
    text = text.replace(f"ใน{target}", "")
    return text


def thai_cleanup(text: str, max_chars: int = 120, keep_end_punct: bool = False) -> str:
    text = (text or "").strip()
    text = ENGLISH_LEFTOVER_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text)
    text = remove_immediate_repetition(text)

    replacements = {
        "รูปภาพของ": "ภาพ",
        "ภาพถ่ายของ": "ภาพ",
        "ซึ่งอยู่ใน": "ใน",
        "ในตอนกลางของ": "กลาง",
        "ในกลาง": "กลาง",
        "ด้านบนของ": "บน",
        "ด้านหน้าของ": "หน้า",
        "ด้านข้างของ": "ข้าง",
        "ที่มีที่มี": "ที่มี",
        " ๆ ๆ": " ๆ",
        "ที่อยู่บน": "บน",
        "ที่อยู่ใน": "ใน",
        "ที่อยู่ข้าง": "ข้าง",
    }
    for src, dst in replacements.items():
        text = text.replace(src, dst)

    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()

    if not keep_end_punct:
        text = text.rstrip(" .,!?:;\n\t")

    if len(text) > max_chars:
        text = text[:max_chars].rstrip(" ,.;:-")

    return text


seed_everything(CFG["SEED"])
device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = (device == "cuda" and CFG["USE_FP16_ON_CUDA"])
print("device:", device, "| use_fp16:", use_fp16)


device: cuda | use_fp16: True


## 3) Load Models

In [ ]:
caption_processor = None
caption_model = None
mt_tokenizer = None
mt_model = None

caption_dtype = torch.float16 if use_fp16 else torch.float32
mt_dtype = torch.float16 if use_fp16 else torch.float32

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def load_caption_model():
    global caption_processor, caption_model
    if caption_model is not None:
        return

    caption_processor = BlipProcessor.from_pretrained(CFG["CAPTION_MODEL"])
    caption_model = BlipForConditionalGeneration.from_pretrained(
        CFG["CAPTION_MODEL"],
        torch_dtype=caption_dtype,
        low_cpu_mem_usage=True,
    ).to(device)
    caption_model.eval()
    print("Loaded caption model:", CFG["CAPTION_MODEL"])


def unload_caption_model():
    global caption_processor, caption_model
    caption_processor = None
    caption_model = None
    cleanup_memory()
    print("Unloaded caption model")


def load_translation_model():
    global mt_tokenizer, mt_model
    if mt_model is not None:
        return

    mt_tokenizer = AutoTokenizer.from_pretrained(CFG["MT_MODEL"])
    mt_model = AutoModelForSeq2SeqLM.from_pretrained(
        CFG["MT_MODEL"],
        torch_dtype=mt_dtype,
        low_cpu_mem_usage=True,
    ).to(device)
    mt_model.eval()
    print("Loaded translation model:", CFG["MT_MODEL"])


def unload_translation_model():
    global mt_tokenizer, mt_model
    mt_tokenizer = None
    mt_model = None
    cleanup_memory()
    print("Unloaded translation model")


## 4) Inference Functions

In [ ]:
@torch.inference_mode()
def _generate_from_inputs(inputs, **generate_kwargs):
    return caption_model.generate(
        **inputs,
        max_new_tokens=CFG["CAPTION_MAX_NEW_TOKENS"],
        min_new_tokens=CFG["CAPTION_MIN_NEW_TOKENS"],
        repetition_penalty=CFG["CAPTION_REPETITION_PENALTY"],
        no_repeat_ngram_size=CFG["CAPTION_NO_REPEAT_NGRAM_SIZE"],
        **generate_kwargs,
    )


def _decode_caption_outputs(output_ids, prompt: str = "") -> List[str]:
    captions = caption_processor.batch_decode(output_ids, skip_special_tokens=True)
    cleaned = []
    prompt_prefix = prompt.strip().lower()
    for caption in captions:
        caption = caption.strip()
        if prompt_prefix and caption.lower().startswith(prompt_prefix):
            caption = caption[len(prompt_prefix):].strip(" :,-")
        cleaned.append(repair_english_caption(caption))
    return cleaned


def _append_candidate_group(candidate_lists: List[List[str]], decoded: List[str], sequences_per_image: int):
    for index, caption in enumerate(decoded):
        image_index = index // sequences_per_image
        if caption:
            candidate_lists[image_index].append(caption)


@torch.inference_mode()
def generate_english_candidates(images: List[Image.Image]) -> List[List[str]]:
    if caption_model is None or caption_processor is None:
        raise RuntimeError("Caption model is not loaded")

    candidate_lists = [[] for _ in images]

    base_inputs = caption_processor(images=images, return_tensors="pt").to(device)
    beam_ids = _generate_from_inputs(
        base_inputs,
        num_beams=CFG["CAPTION_NUM_BEAMS"],
        num_return_sequences=1,
        length_penalty=CFG["CAPTION_LENGTH_PENALTY"],
    )
    _append_candidate_group(candidate_lists, _decode_caption_outputs(beam_ids), 1)

    if CFG["USE_SAMPLING_CANDIDATES"]:
        sample_ids = _generate_from_inputs(
            base_inputs,
            do_sample=True,
            top_p=CFG["CAPTION_TOP_P"],
            temperature=CFG["CAPTION_TEMPERATURE"],
            num_beams=1,
            num_return_sequences=CFG["CAPTION_NUM_RETURN_SEQUENCES"],
            length_penalty=1.0,
        )
        _append_candidate_group(
            candidate_lists,
            _decode_caption_outputs(sample_ids),
            CFG["CAPTION_NUM_RETURN_SEQUENCES"],
        )

    if CFG["USE_PROMPT_CANDIDATE"] and CFG["CAPTION_PROMPT"].strip():
        prompt = CFG["CAPTION_PROMPT"].strip()
        prompt_inputs = caption_processor(
            images=images,
            text=[prompt] * len(images),
            return_tensors="pt",
            padding=True,
        ).to(device)
        prompt_ids = _generate_from_inputs(
            prompt_inputs,
            num_beams=max(3, CFG["CAPTION_NUM_BEAMS"] - 1),
            num_return_sequences=1,
            length_penalty=1.0,
        )
        _append_candidate_group(candidate_lists, _decode_caption_outputs(prompt_ids, prompt=prompt), 1)

    final_candidates = []
    for candidates in candidate_lists:
        deduped = []
        seen = set()
        for caption in candidates:
            normalized = repair_english_caption(caption)
            if normalized and normalized not in seen:
                seen.add(normalized)
                deduped.append(normalized)
        final_candidates.append(deduped)

    return final_candidates


def select_best_english_caption(image: Image.Image, candidates: List[str]) -> str:
    del image
    if not candidates:
        return ""
    ranked = sorted(candidates, key=lambda text: (caption_quality_score(text), len(text)), reverse=True)
    return ranked[0]


def normalize_english_for_mt(text: str) -> str:
    text = repair_english_caption(text)
    text = re.sub(r"\ba close up of\b", "close-up of", text)
    return text.strip()


@torch.inference_mode()
def translate_batch_en_to_th(texts: List[str]) -> List[str]:
    if mt_model is None or mt_tokenizer is None:
        raise RuntimeError("Translation model is not loaded")
    if not texts:
        return []

    mt_tokenizer.src_lang = CFG["SRC_LANG"]
    batch = mt_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    )
    batch = {key: value.to(device) for key, value in batch.items()}

    output_ids = mt_model.generate(
        **batch,
        forced_bos_token_id=mt_tokenizer.convert_tokens_to_ids(CFG["TGT_LANG"]),
        max_new_tokens=CFG["MT_MAX_NEW_TOKENS"],
        num_beams=CFG["MT_NUM_BEAMS"],
        length_penalty=CFG["MT_LENGTH_PENALTY"],
        no_repeat_ngram_size=CFG["MT_NO_REPEAT_NGRAM_SIZE"],
    )
    return mt_tokenizer.batch_decode(output_ids, skip_special_tokens=True)


def post_edit_thai(text_en: str, text_th: str) -> str:
    text_en = (text_en or "").lower()
    text_th = (text_th or "").strip()

    if not text_th:
        return CFG["FALLBACK_THAI_CAPTION"]

    if "buddha" in text_en and "statue" in text_en:
        for bad in ["เหรียญพระพุทธรูป", "เหรียญพระพุทธเจ้า", "รูปปั้นของพระพุทธรูป"]:
            text_th = text_th.replace(bad, "พระพุทธรูป")
        text_th = text_th.replace("รูปปั้นพระพุทธรูป", "พระพุทธรูป")
        text_th = text_th.replace("พระพุทธรูปทองคํา", "พระพุทธรูปทองคำ")
    elif "statue" in text_en or "sculpture" in text_en:
        text_th = re.sub(r"เหรียญ(?:ปั้น|สักรูป)?", "รูปปั้น", text_th)
        text_th = text_th.replace("ประติมากรรม", "รูปปั้น")

    if "frog" in text_en:
        for bad in ["หนอน", "พยาธิ"]:
            text_th = text_th.replace(bad, "กบ")
    if "goat" in text_en:
        for bad in ["แมว", "หมู"]:
            text_th = text_th.replace(bad, "แพะ")
    if "beetle" in text_en:
        for bad in ["หนอน", "พยาธิ", "แมลงกระรอก"]:
            text_th = text_th.replace(bad, "ด้วง")
    if "squirrel" in text_en:
        for bad in ["หมึก", "ปลาหมึก"]:
            text_th = text_th.replace(bad, "กระรอก")
    if "wasp" in text_en:
        for bad in ["ราหมี", "หมี"]:
            text_th = text_th.replace(bad, "ตัวต่อ")
    if "plate" in text_en:
        text_th = text_th.replace("ถาด", "จาน")
        text_th = collapse_duplicate_tail(text_th, "จาน")
    if "bowl" in text_en:
        bowl_word = "ชาม" if is_food_caption(text_en) else "ถ้วย"
        text_th = text_th.replace("กระปุก", bowl_word)
        text_th = text_th.replace("ถัง", bowl_word)
        text_th = collapse_duplicate_tail(text_th, bowl_word)
    if "basket" in text_en:
        text_th = text_th.replace("กระเป๋าสตางค์", "ตะกร้า")
    if "mushroom" in text_en:
        text_th = text_th.replace("เห็ดที่นั่งอยู่บน", "เห็ดบน")
    if "bird" in text_en:
        text_th = text_th.replace("นกที่นั่งอยู่บน", "นกบน")

    if not any(word in text_en for word in HUMAN_WORDS):
        replacements = {
            "ที่นั่งอยู่บน": "บน",
            "นั่งอยู่บน": "อยู่บน",
            "ที่นั่งบน": "บน",
            "นั่งบน": "บน",
            "ที่นอนอยู่บน": "บน",
            "ที่นอนอยู่ใน": "ใน",
            "นอนอยู่ใน": "อยู่ใน",
        }
        for src, dst in replacements.items():
            text_th = text_th.replace(src, dst)

    text_th = text_th.replace("ไม้ยาง", "ไม้ไผ่") if "bamboo" in text_en else text_th
    text_th = text_th.replace("องค์ประกอบของน้ํา", "แหล่งน้ำ") if "body of water" in text_en else text_th
    text_th = text_th.replace("เครื่องเย็บ", "เครื่องทอ") if "weaving" in text_en else text_th
    text_th = text_th.replace("น้ํ", "น้ำ")
    text_th = text_th.replace("คํ", "คำ")

    text_th = thai_cleanup(
        text_th,
        max_chars=CFG["MAX_THAI_CHARS"],
        keep_end_punct=CFG["KEEP_END_PUNCT"],
    )

    if len(text_th) < CFG["MIN_THAI_CHARS"]:
        return CFG["FALLBACK_THAI_CAPTION"]

    return text_th


## 5) Run Captioning

In [ ]:
test_dir = Path(CFG["TEST_DIR"])
image_files = sorted([p for p in test_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])


def load_images(paths: List[Path]) -> List[Image.Image]:
    images = []
    for path in paths:
        with Image.open(path) as img:
            images.append(img.convert("RGB"))
    return images


def caption_batch(paths: List[Path]) -> List[Dict[str, str]]:
    images = load_images(paths)
    candidate_groups = generate_english_candidates(images)
    rows = []
    for image_path, image, candidates in zip(paths, images, candidate_groups):
        best_caption = select_best_english_caption(image, candidates)
        rows.append({
            "image_filename": image_path.name,
            "caption_en": best_caption,
        })
    return rows


def caption_batch_with_retry(paths: List[Path]) -> List[Dict[str, str]]:
    if not paths:
        return []
    try:
        return caption_batch(paths)
    except RuntimeError as error:
        if "out of memory" in str(error).lower() and len(paths) > 1:
            cleanup_memory()
            mid = len(paths) // 2
            return caption_batch_with_retry(paths[:mid]) + caption_batch_with_retry(paths[mid:])

        cleanup_memory()
        return [{
            "image_filename": path.name,
            "caption_en": "",
        } for path in paths]


def translate_batch_with_retry(texts: List[str]) -> List[str]:
    if not texts:
        return []
    try:
        return translate_batch_en_to_th(texts)
    except RuntimeError as error:
        if "out of memory" in str(error).lower() and len(texts) > 1:
            cleanup_memory()
            mid = len(texts) // 2
            return translate_batch_with_retry(texts[:mid]) + translate_batch_with_retry(texts[mid:])
        cleanup_memory()
        return [CFG["FALLBACK_THAI_CAPTION"] for _ in texts]


# Phase 1: caption images in English
load_caption_model()
caption_rows = []
caption_batches = chunked(image_files, CFG["CAPTION_BATCH_SIZE"])
num_caption_batches = (len(image_files) + CFG["CAPTION_BATCH_SIZE"] - 1) // CFG["CAPTION_BATCH_SIZE"]

for batch_index, paths in enumerate(tqdm(caption_batches, total=num_caption_batches, desc="Caption phase"), start=1):
    batch_rows = caption_batch_with_retry(paths)
    caption_rows.extend(batch_rows)

    if batch_rows and (batch_index % max(1, CFG["BATCH_PRINT_EVERY"]) == 0):
        sample = batch_rows[-1]
        print(f"[caption {len(caption_rows)}] {sample['image_filename']} | EN: {sample['caption_en']}")

caption_df = pd.DataFrame(caption_rows)
unload_caption_model()

# Phase 2: translate to Thai in batches
load_translation_model()
english_for_mt = [normalize_english_for_mt(text) for text in caption_df["caption_en"].fillna("")]
thai_predictions = []

for start in tqdm(range(0, len(english_for_mt), CFG["TRANSLATE_BATCH_SIZE"]), desc="Translation phase"):
    batch_en = english_for_mt[start:start + CFG["TRANSLATE_BATCH_SIZE"]]
    safe_batch = [text if text else "an object in a scene" for text in batch_en]
    raw_batch_th = translate_batch_with_retry(safe_batch)

    for source_en, raw_th in zip(batch_en, raw_batch_th):
        if not source_en:
            thai_predictions.append(CFG["FALLBACK_THAI_CAPTION"])
        else:
            thai_predictions.append(post_edit_thai(source_en, raw_th))

unload_translation_model()

pred_df = caption_df.copy()
pred_df["caption_th"] = thai_predictions
pred_df["caption_th"] = pred_df["caption_th"].fillna(CFG["FALLBACK_THAI_CAPTION"])
pred_df.loc[pred_df["caption_th"].str.len() < CFG["MIN_THAI_CHARS"], "caption_th"] = CFG["FALLBACK_THAI_CAPTION"]
pred_df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded translation model: facebook/nllb-200-distilled-1.3B


Translation phase:   0%|          | 0/42 [00:00<?, ?it/s]

Unloaded translation model


,image_filename,caption_en,caption_th
0,00000.jpg,a brown and white horse walking down a dirt road,ม้าสีน้ำาตาลและขาวเดินลงทางดิน
1,00001.jpg,a plate of food with meat and vegetables on it,จานอาหารที่มีเนื้อและผัก
2,00002.jpg,a small white flower on a green leaf,ดอกไม้สีขาวเล็ก ๆ บนใบสีเขียว
3,00003.jpg,a close up of a cow's nose,ภาพใกล้ๆ ของจมูกวัว
4,00004.jpg,a small dog sitting on top of a table next to ...,สุนัขตัวเล็กๆ อยู่บนโต๊ะข้างเคียงกับถ้วยของหวาน


## 6) Quick Quality Check

In [ ]:
display(pred_df.sample(min(10, len(pred_df)), random_state=CFG["SEED"]))

trailing_stopword_count = pred_df["caption_en"].fillna("").str.contains(
    r"\b(?:a|an|the|of|with|and|in|on|at|to|from|by|for|near)$",
    case=False,
    regex=True,
).sum()
english_leftover_count = pred_df["caption_th"].fillna("").str.contains(r"[A-Za-z]", regex=True).sum()
fallback_count = (pred_df["caption_th"] == CFG["FALLBACK_THAI_CAPTION"]).sum()

print("trailing_stopword_count:", int(trailing_stopword_count))
print("english_leftover_count:", int(english_leftover_count))
print("fallback_count:", int(fallback_count))


,image_filename,caption_en,caption_th
1860,01860.jpg,a small animal that is laying on the ground,สัตว์เล็กๆ บนพื้น
353,00353.jpg,a man walking down a dirt road with trees in t...,ชายคนหนึ่งเดินลงทางดินที่มีต้นไม้อยู่เบื้องหลัง
1333,01333.jpg,a bird is perched on a branch in the forest,นกอยู่บนสาขาในป่า
905,00905.jpg,a white flower with green leaves in the backgr...,ดอกไม้สีขาวที่มีใบเขียวในพื้นหลัง
1289,01289.jpg,a small flower in the middle of a field,ดอกไม้เล็ก ๆ กลางสนาม
1273,01273.jpg,a woman in a wet suit and snoring gear with a ...,ผู้หญิงในชุดชื้นและอุปกรณ์หงุดหงิดพร้อมกับสายพ...
938,00938.jpg,a field of banana trees with blue sky in the b...,สนามต้นกล้วยไม้ที่มีท้องฟ้าสีน้ำาเงินอยู่เบื้อ...
1731,01731.jpg,a body of water with trees in the background,น้ำาที่มีต้นไม้อยู่เบื้องหลัง
65,00065.jpg,a trail in the woods with trees and leaves on ...,เส้นทางในป่าที่มีต้นไม้และใบไม้อยู่บนพื้น
1323,01323.jpg,a view of a river with trees and mountains in ...,มุมมองของแม่น้ำาที่มีต้นไม้และภูเขาในเบื้องหลัง


trailing_stopword_count: 0
english_leftover_count: 1
fallback_count: 0


## 7) Build Submission

In [ ]:
submission = pd.read_csv(CFG["SAMPLE_SUBMISSION_PATH"], dtype=str)
pred_df = pred_df.copy()
pred_df["image_id"] = pred_df["image_filename"].map(lambda name: Path(name).stem)
pred_map = dict(zip(pred_df["image_id"], pred_df["caption_th"]))

submission["image_id"] = submission["image_id"].astype(str).str.zfill(5)

if CFG["OVERWRITE_SUBMISSION"]:
    submission["caption"] = submission["image_id"].map(lambda image_id: pred_map.get(image_id, CFG["FALLBACK_THAI_CAPTION"]))
else:
    for idx, row in submission.iterrows():
        image_id = str(row["image_id"])
        if (pd.isna(row["caption"]) or str(row["caption"]).strip() == "") and image_id in pred_map:
            submission.at[idx, "caption"] = pred_map[image_id]
    submission["caption"] = submission["caption"].fillna(CFG["FALLBACK_THAI_CAPTION"])

assert len(submission) == len(pd.read_csv(CFG["SAMPLE_SUBMISSION_PATH"], dtype=str))
submission.to_csv(CFG["OUTPUT_PATH"], index=False, encoding="utf-8")
print("Saved:", CFG["OUTPUT_PATH"])
submission.head()


Saved: submission_balanced_colab.csv


,image_id,caption
0,01354,รูปภาพใกล้ๆ ของปลาแครบขนาดเล็ก
1,01413,หนอนเล็กๆ บนหิน
2,01802,เกาะที่อยู่กลางน้ำา
3,01243,ลิงนั่งบนพื้นข้างผนัง
4,00693,หมึกสองตัวขี่บนหินในป่า


In [ ]:
# Optional: evaluate on a local validation split if caption labels are available in the competition input.
from pathlib import Path


def discover_caption_csvs(root: Path):
    candidates = []
    for csv_path in root.rglob("*.csv"):
        try:
            sample = pd.read_csv(csv_path, nrows=3)
        except Exception:
            continue
        lowered = {column.lower() for column in sample.columns}
        if "caption" in lowered and ({"image_id", "image_filename"} & lowered):
            candidates.append(csv_path)
    return candidates


competition_root = Path(CFG["TEST_DIR"]).resolve().parents[1]
caption_csvs = discover_caption_csvs(competition_root)

if not caption_csvs:
    print("No caption CSV with labels found. Skip BLEU evaluation cell.")
else:
    print("Candidate label files:")
    for path in caption_csvs[:10]:
        print(" -", path)
    print("Use one of these files to build a held-out split and compare BLEU with sacrebleu.")


Candidate label files:
 - /root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/sample_submission.csv
Use one of these files to build a held-out split and compare BLEU with sacrebleu.
